# A6: Naive RAG vs Contextual Retrieval  
## Student ID: 125983  
## Assigned Chapter: 3 — *N-gram Language Models*

### Objective
This notebook builds a domain-specific QA system for **Chapter 3** and compares:

1. **Naive RAG**
2. **Contextual Retrieval**

The assignment requires:
- chapter-based source preparation
- at least 20 QA pairs
- evaluation with **ROUGE-1**, **ROUGE-2**, and **ROUGE-L**
- a chatbot app that uses **Contextual Retrieval**

## Models used

### Retriever
- Primary: `sentence-transformers/all-MiniLM-L6-v2`
- Fallback: TF-IDF cosine similarity

### Generator
- Default: local extractive sentence-ranking answerer
- Optional: `gpt-4o-mini` if `OPENAI_API_KEY` is available

> This submission is fully runnable without an API key. The optional API path is only an extra enhancement.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT / "src"))

from rag_pipeline import (
    answer_question,
    build_retriever_for_mode,
    evaluate_rows,
    extract_pages_and_sections,
    resolve_pdf_path,
)

PDF_PATH = resolve_pdf_path()
PDF_PATH

'data/chapter3_ngram_language_models.pdf'

## 1) Source discovery and data preparation
The assigned source is Chapter 3 of *Speech and Language Processing*. The PDF is bundled inside the repository, so the notebook can run from the local file first and only download the chapter if the PDF is missing.

In [2]:
pages = extract_pages_and_sections(PDF_PATH)
len(pages), pages[0]["section"], pages[0]["clean_text"][:200]

(26,
 '3.0 Introduction',
 'Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER 3 N-gram Language Models “You are uniformly charming!” cried')

## 2) Build the two retrieval pipelines
- **Naive RAG** indexes plain chunks.
- **Contextual Retrieval** prepends a short context prefix containing the chapter title, section, page number, and a deterministic chunk summary before indexing.

In [3]:
naive_chunks, naive_retriever = build_retriever_for_mode(mode="naive", pdf_path=PDF_PATH)
contextual_chunks, contextual_retriever = build_retriever_for_mode(mode="contextual", pdf_path=PDF_PATH)

{
    "naive_chunk_count": len(naive_chunks),
    "naive_retriever_mode": naive_retriever.mode,
    "contextual_chunk_count": len(contextual_chunks),
    "contextual_retriever_mode": contextual_retriever.mode,
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'naive_chunk_count': 91,
 'naive_retriever_mode': 'sentence-transformer',
 'contextual_chunk_count': 91,
 'contextual_retriever_mode': 'sentence-transformer'}

In [4]:
# Peek at a contextualized chunk
contextual_chunks[0].contextual_text[:800]

'This chunk from Chapter 3 – N-gram Language Models, 3.0 Introduction, page 1, discusses Speech and Language Processing.\n\nSpeech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER 3 N-gram Language Models “You are uniformly charming!” cried he, with a smile of associating and now and then I bowed and they perceived a chaise and four to wish for. Random sentence generated from a Jane Austen trigram model Predicting is difﬁcult—especially about the future, as the old quip goes. But how about predicting something that seems much easier, like the next word someone is going to say? What word, for example, is likely to follow The water of Walden Pond is so beautifully ... You might conclude that a likely word is blu'

## 3) QA pair dataset
The following 20 questions were written strictly from Chapter 3. Ground-truth answers are concise reference answers used for evaluation.

In [5]:
qa_pairs = [
  {
    "question": "What is a language model according to Chapter 3?",
    "ground_truth_answer": "A language model is a machine learning model that predicts upcoming words. More formally, it assigns a probability distribution over possible next words and can also assign a probability to an entire sentence."
  },
  {
    "question": "What is an n-gram?",
    "ground_truth_answer": "An n-gram is a sequence of n words. For example, a bigram has two words and a trigram has three words, and n-gram models estimate the probability of a word from the previous n−1 words."
  },
  {
    "question": "Why can’t we estimate next-word probabilities using full sentence histories directly?",
    "ground_truth_answer": "Because language is creative and even huge corpora do not contain reliable counts for every full sentence or very long history. Many contexts never occur, so direct counting gives poor estimates."
  },
  {
    "question": "How does the chain rule help compute sentence probability?",
    "ground_truth_answer": "The chain rule decomposes the probability of a word sequence into a product of conditional probabilities, where each word is conditioned on all previous words in the sequence."
  },
  {
    "question": "What is the Markov assumption in a bigram model?",
    "ground_truth_answer": "The Markov assumption says that instead of conditioning on the entire history, a bigram model approximates the next word probability using only the previous word."
  },
  {
    "question": "How is an n-gram probability estimated with maximum likelihood estimation (MLE)?",
    "ground_truth_answer": "MLE estimates an n-gram probability by dividing the count of the full n-gram by the count of its prefix context. This is a relative-frequency estimate obtained by normalizing corpus counts."
  },
  {
    "question": "Why are the special symbols <s> and </s> added in the chapter’s examples?",
    "ground_truth_answer": "The start symbol <s> provides context for the first word of a sentence, and the end symbol </s> lets the model represent sentence endings properly. The end symbol is needed so the model forms a true probability distribution over sentences."
  },
  {
    "question": "Why must language models be evaluated on separate training and test sets?",
    "ground_truth_answer": "A separate test set checks how well the model generalizes to unseen data rather than memorizing the training corpus. Evaluating on training data alone can give misleadingly optimistic results."
  },
  {
    "question": "What is perplexity?",
    "ground_truth_answer": "Perplexity is the inverse probability of the test set, normalized by the number of words or tokens. It is a standard intrinsic evaluation metric for language models."
  },
  {
    "question": "Why is lower perplexity better?",
    "ground_truth_answer": "Lower perplexity means the model assigns higher probability to the observed test sequence, so it is a better predictor of the test data."
  },
  {
    "question": "What does perplexity mean as a weighted average branching factor?",
    "ground_truth_answer": "Perplexity can be interpreted as the weighted average branching factor of a language: the effective average number of choices the model faces when predicting the next word, weighted by their probabilities."
  },
  {
    "question": "How does sampling from a language model work?",
    "ground_truth_answer": "Sampling generates text by choosing words randomly according to the probability distribution defined by the model. More probable words are selected more often than less probable ones."
  },
  {
    "question": "Why can larger n-grams lead to overfitting?",
    "ground_truth_answer": "As n increases, the model matches the training corpus more closely and encodes more corpus-specific facts, but it generalizes less well to unseen text. This makes larger n-grams more prone to overfitting."
  },
  {
    "question": "What problem does smoothing solve in n-gram language models?",
    "ground_truth_answer": "Smoothing addresses zero-probability n-grams by moving some probability mass from frequent events to unseen events. This prevents unseen but possible sequences from receiving probability zero."
  },
  {
    "question": "What is Laplace smoothing?",
    "ground_truth_answer": "Laplace smoothing, also called add-one smoothing, adds one to every count before normalizing into probabilities. It introduces nonzero probabilities for unseen events but is usually too crude for modern language modeling."
  },
  {
    "question": "What is add-k smoothing?",
    "ground_truth_answer": "Add-k smoothing is a generalization of add-one smoothing that adds a fractional value k to each count instead of 1. It moves less probability mass to unseen events, but it still tends to perform poorly for language modeling."
  },
  {
    "question": "How does interpolation help with unseen or unreliable higher-order n-grams?",
    "ground_truth_answer": "Interpolation combines trigram, bigram, and unigram probabilities with weights, so when higher-order counts are sparse or missing the model can still rely on lower-order estimates."
  },
  {
    "question": "How are the lambda weights in interpolation chosen?",
    "ground_truth_answer": "The lambda weights are learned from a held-out corpus by choosing values that maximize the likelihood of that held-out data. The chapter mentions the EM algorithm as one way to find them."
  },
  {
    "question": "What is stupid backoff?",
    "ground_truth_answer": "Stupid backoff is a backoff method that uses the highest-order n-gram count when available and otherwise backs off to a lower-order model, multiplying by a constant factor. It is simple and effective for large-scale settings but does not produce a true probability distribution."
  },
  {
    "question": "Why must two language models use the same vocabulary if we want to compare their perplexities fairly?",
    "ground_truth_answer": "Perplexity values are only comparable when the models use identical vocabularies, because vocabulary differences change which tokens are considered known or unknown and alter the probability space being evaluated."
  }
]
len(qa_pairs), qa_pairs[0]

(20,
 {'question': 'What is a language model according to Chapter 3?',
  'ground_truth_answer': 'A language model is a machine learning model that predicts upcoming words. More formally, it assigns a probability distribution over possible next words and can also assign a probability to an entire sentence.'})

## 4) Run the comparison experiment
For reproducibility, the notebook uses the **extractive generator** by default. If an `OPENAI_API_KEY` is set, the same pipeline can optionally use `gpt-4o-mini`

In [6]:
def run_batch(qa_pairs, chunks, retriever, answer_key, top_k=3, generator="extractive"):
    rows = []
    for item in qa_pairs:
        result = answer_question(
            query=item["question"],
            chunks=chunks,
            retriever=retriever,
            top_k=top_k,
            generator=generator,
        )
        rows.append(
            {
                "question": item["question"],
                "ground_truth_answer": item["ground_truth_answer"],
                answer_key: result["answer"],
            }
        )
    return rows

naive_rows = run_batch(qa_pairs, naive_chunks, naive_retriever, "naive_rag_answer")
context_rows = run_batch(qa_pairs, contextual_chunks, contextual_retriever, "contextual_retrieval_answer")

merged_rows = []
for left, right in zip(naive_rows, context_rows):
    merged_rows.append({
        **left,
        **{k: v for k, v in right.items() if k not in {"question", "ground_truth_answer"}},
    })

len(merged_rows), merged_rows[0]

(20,
 {'question': 'What is a language model according to Chapter 3?',
  'ground_truth_answer': 'A language model is a machine learning model that predicts upcoming words. More formally, it assigns a probability distribution over possible next words and can also assign a probability to an entire sentence.',
  'naive_rag_answer': 'N-gram models can be trained by counting in a training corpus and normalizing the counts (the maximum likelihood estimate). • N-gram language models can be evaluated on a test set using perplexity. • The perplexity of a test set according to a language model is a function of the probability of the test set: the inverse test set probability according to the model, normalized by the length. • Sampling from a language model means to generate some sentences, choosing each sentence according to its likelihood as .wN)− 1 N = N √ 1 P(w1w2 . . .wN) 3.8 Summary This chapter introduced language modeling via the n-gram model, a classic model that allows us to introduce m

In [7]:
results_df = pd.DataFrame(merged_rows)
results_df.head(3)

,question,ground_truth_answer,naive_rag_answer,contextual_retrieval_answer
0,What is a language model according to Chapter 3?,A language model is a machine learning model t...,N-gram models can be trained by counting in a ...,N-gram models can be trained by counting in a ...
1,What is an n-gram?,An n-gram is a sequence of n words. For exampl...,It is also common to prune n-gram language mod...,An n-gram is a sequence of n words: a 2-gram (...
2,Why can’t we estimate next-word probabilities ...,Because language is creative and even huge cor...,"In other words, instead of computing the proba...","For this reason, we’ll need more clever ways t..."


In [8]:
naive_scores = evaluate_rows(merged_rows, "naive_rag_answer")
context_scores = evaluate_rows(merged_rows, "contextual_retrieval_answer")

evaluation_table = pd.DataFrame(
    [
        ["Naive RAG", naive_scores["ROUGE-1"], naive_scores["ROUGE-2"], naive_scores["ROUGE-L"]],
        ["Contextual Retrieval", context_scores["ROUGE-1"], context_scores["ROUGE-2"], context_scores["ROUGE-L"]],
    ],
    columns=["Method", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
)

evaluation_table

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.218775,0.067009,0.141819
1,Contextual Retrieval,0.223747,0.076243,0.153375


## 5) Save the deliverables 
This cell writes the JSON comparison file and the ROUGE summary artifacts into the `answer/` folder.

In [9]:
answer_dir = PROJECT_ROOT / "answer"
answer_dir.mkdir(exist_ok=True)

with open(answer_dir / "response-st-125983-chapter-3.json", "w", encoding="utf-8") as f:
    json.dump(merged_rows, f, indent=2, ensure_ascii=False)

with open(answer_dir / "response-st125983-chapter-3.json", "w", encoding="utf-8") as f:
    json.dump(merged_rows, f, indent=2, ensure_ascii=False)

rouge_summary = {
    "naive_rag": naive_scores,
    "contextual_retrieval": context_scores,
}
with open(answer_dir / "rouge_summary.json", "w", encoding="utf-8") as f:
    json.dump(rouge_summary, f, indent=2, ensure_ascii=False)

evaluation_table.to_csv(answer_dir / "evaluation_table.csv", index=False)

sorted(p.name for p in answer_dir.iterdir())

['evaluation_table.csv',
 'response-st-125983-chapter-3.json',
 'response-st125983-chapter-3.json',
 'rouge_summary.json']

## 6) Discussion
Contextual Retrieval performs slightly better than Naive RAG on all three ROUGE metrics. The improvement is smaller than expected, but it is consistent across ROUGE-1, ROUGE-2, and ROUGE-L. This suggests that adding contextual prefixes helps retrieval identify somewhat more relevant chunks, especially when similar terms appear in different parts of the chapter.

In [10]:
example = answer_question(
    query="Why is lower perplexity better?",
    chunks=contextual_chunks,
    retriever=contextual_retriever,
    top_k=3,
    generator="extractive",
)
example

{'answer': 'So a lower perplexity tells us that a language model is a better predictor of the test set. And the higher the probability, the lower the perplexity (since as Eq. 3.15 showed, perplexity is related inversely to the probability of the test sequence according to the model). And the perplexity of two language models is only comparable if they use identical vocabularies.',
 'sources': [{'chunk_id': 'p10_c3',
   'page': 10,
   'section': '3.3.1 Perplexity as Weighted Average Branching Factor',
   'score': 0.5511,
   'text': 'next, and so it assigns them a higher probability. And the higher the probability, the lower the perplexity (since as Eq. 3.15 showed, perplexity is related inversely to the probability of the test sequence according to the model). So a lower perplexity tells us that a language model is a better predictor of the test set. Note that in computing perplexities, the language model must be constructed without any knowledge of the test set, or else the perplexity 

## 7) Web app note
The `app/app.py` Streamlit interface uses the **Contextual Retrieval** backend from this notebook and displays the answer together with the cited source chunks, matching the assignment requirement.